In [1]:
import torch
import torch.nn.functional as F

# Data Preparation

In [2]:
names = []
with open("./data/names.txt", "r") as f:
    for line in f:
        names.append(line.rstrip())

print(len(names))
names[:5]

32033


['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [3]:
def create_bigrams(words):
    bigrams = []

    for word in words:
        for c1, c2 in zip(word, word[1:]):
            bigrams.append(c1 + c2)

    return bigrams 

create_bigrams(["rahul"])

['ra', 'ah', 'hu', 'ul']

In [4]:
def pad_words(words):
    padded = []

    for word in words:
        padded.append("".join([".", word, "."]))

    return padded

pad_words(["rahul"])

['.rahul.']

In [5]:
alphabet = ['.'] + sorted(set("".join(names)))
alphabet_len = len(alphabet)

print(alphabet)
print(alphabet_len)

['.', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
27


In [6]:
char_to_idx = {}
for idx, char in enumerate(alphabet):
    char_to_idx[char] = idx

char_to_idx

{'.': 0,
 'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26}

In [7]:
padded_names = pad_words(names)
padded_names[:5]

['.emma.', '.olivia.', '.ava.', '.isabella.', '.sophia.']

In [8]:
bigrams = create_bigrams(padded_names)
bigrams[:12]

['.e', 'em', 'mm', 'ma', 'a.', '.o', 'ol', 'li', 'iv', 'vi', 'ia', 'a.']

In [9]:
xs = []
ys = []
for bigram in bigrams:
    xs.append(char_to_idx[bigram[0]])
    ys.append(char_to_idx[bigram[1]])

xs[:5]

[0, 5, 13, 13, 1]

# Counting model

In [10]:
generator = torch.Generator().manual_seed(2774943997)

In [11]:
counts = torch.zeros(alphabet_len, alphabet_len)
counts.shape

torch.Size([27, 27])

In [12]:
for x, y in zip(xs, ys):
    counts[x][y] += 1

counts[0][5]

tensor(1531.)

In [13]:
def normalize(tensor):
    row_sums = torch.sum(tensor, dim=1, keepdim=True)

    probs = tensor / row_sums
    assert torch.allclose(probs.sum(dim=1), torch.ones(len(tensor)))

    return probs

In [14]:
probs = normalize(counts + 1)

In [15]:
probs.shape

torch.Size([27, 27])

In [16]:
def generate_names(num, probs_matrix, start_idx):
    words = []
    for _ in range(num):
        word = ""
        next_idx = start_idx
        while True:
            vector = probs_matrix[next_idx]
            idx = torch.multinomial(vector, num_samples=1, replacement=True, generator=generator).item()
            char = alphabet[idx]
            if char == ".":
                break
      
            word += char
            next_idx = idx

        words.append(word)

    return words

generate_names(5, probs, char_to_idx['.'])

['kst', 'allallle', 'susharyn', 'lorist', 'stis']

In [17]:
loss = 0.0
for x, y in zip(xs, ys):
    prob = probs[x][y]
    loss += torch.log(prob)

loss = -(loss / len(bigrams))
loss

tensor(2.4544)

# Neural Network implementation

In [18]:
weights = torch.randn(alphabet_len, alphabet_len, requires_grad=True)

In [19]:
x_oh = F.one_hot(torch.tensor(xs), num_classes=alphabet_len).float()
x_oh.shape

torch.Size([228146, 27])

In [20]:
lr = 80
num_epochs = 10000

for epoch in range(num_epochs):
    weights.grad = None

    logits = x_oh @ weights
    nn_preds = normalize(logits.exp())

    # Add normalization to the loss
    loss = -torch.log(nn_preds[torch.arange(len(ys)), ys]).mean() + 0.0001 * (weights**2).mean()
    loss.backward()

    if epoch % 1000 == 0:
        print(f"{epoch=}: loss={loss.item():4f}")

    with torch.no_grad():
        weights -= lr * weights.grad


epoch=0: loss=3.665754
epoch=1000: loss=2.460099
epoch=2000: loss=2.459756
epoch=3000: loss=2.459658
epoch=4000: loss=2.459614
epoch=5000: loss=2.459590
epoch=6000: loss=2.459576
epoch=7000: loss=2.459566
epoch=8000: loss=2.459559
epoch=9000: loss=2.459555


In [21]:
nn_probs = normalize(weights.exp())
generate_names(5, nn_probs, char_to_idx['.'])

['naitan', 'ker', 'k', 'a', 'jaisariniaisoha']

In [22]:
(probs - nn_probs).abs().max()

tensor(0.0945, grad_fn=<MaxBackward1>)